# imports

In [ ]:
import joblib  # joblib saves/loads scikit-learn models to/from a file
import time  # to measure how long training takes
from pathlib import Path

import matplotlib.pyplot as plt  # matplotlib = library for drawing plots and images
import numpy as np  # numpy = library for working with arrays of numbers
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestClassifier  # scikit-learn's Random Forest
from tensorflow.keras.datasets import mnist  # Keras loads MNIST and caches the download on first use.
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split  # scikit-learn's tool for splitting data

# Render saved figures inline when executed in a notebook kernel.
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('matplotlib', 'inline')

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)  # Python, NumPy, and TensorFlow seeds
tf.config.experimental.enable_op_determinism()

PROJECT_ROOT = Path.cwd().resolve()
MODEL_DIR = PROJECT_ROOT / 'models'
FIGURE_DIR = PROJECT_ROOT / 'report_assets'
(PROJECT_ROOT / 'results').mkdir(exist_ok=True)
for folder in (MODEL_DIR, FIGURE_DIR):
    folder.mkdir(exist_ok=True)

# 2. Load the dataset
#    It returns TWO pairs: (training images, training labels) and (test images, test labels)

In [ ]:
(X_train_full, y_train_full), (X_test_raw, y_test) = mnist.load_data()

# 3. Look at the shapes (how many images, and what size)
# .shape= (numb of image, rows, colomns)

In [ ]:
print("X_train_full:", X_train_full.shape)
print("y_train_full:", y_train_full.shape)
print("X_test:      ", X_test_raw.shape)
print("y_test:      ", y_test.shape)

# 4. Look at the numbers inside one image

In [ ]:
print("Smallest pixel value:", X_train_full.min())
print("Largest pixel value: ", X_train_full.max())
print("Label of the first image:", y_train_full[0])

# ba test el labels (print("Label of the first image:", y_train_full[0])) by2ol el label bta3 kol sora which is the image drawn

In [ ]:
import matplotlib.pyplot as plt          # matplotlib = library for drawing plots and images

plt.imshow(X_train_full[2005], cmap="gray") # imshow = "image show"; cmap="gray" draws it in black-and-white
plt.title("Label: " + str(y_train_full[2005]))  # put the label above the picture
plt.axis("off")                          # hide the x/y axis numbers
plt.show()                               # display it

# mn el akher by3ml split le 10000 soora (shuffled using train_test_split) 3shan n3ml validation beeha ba3din

In [ ]:
# Split the 60,000 training images into 50,000 train + 10,000 validation.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_full, y_train_full,  # the data to split (images and their labels together)
    test_size=10000,            # how many go into the second part (validation)
    random_state=SEED,         # fixed seed -> same shuffle every time, for everyone
    stratify=y_train_full,     # keep the same share of each digit in both parts
)

print('Train:', X_train_raw.shape, y_train.shape)
print('Validation:', X_val_raw.shape, y_val.shape)
print('Test:', X_test_raw.shape, y_test.shape)

# 1. Normalize: divide every pixel by 255 so values go from 0-255 to 0.0-1.0
#    astype("float32") explicitly chooses the dtype and reduces memory; NumPy true division already preserves decimals.

- .astype("float32") converts whole-number pixels into decimal numbers.

In [ ]:
# Normalize to 0-1 using float32.
X_train = X_train_raw.astype('float32') / 255.0
X_val = X_val_raw.astype('float32') / 255.0
X_test = X_test_raw.astype('float32') / 255.0

# 2. Flattened version (for Random Forest and MLP): each 28x28 image becomes one row of 784 numbers

- .reshape(...) rearranges the same numbers into a new shape without changing them.

- In reshape(len(X_train), 784), len(X_train) keeps the number of images (50,000). 784 is 28 × 28, so each image becomes one row.

In [ ]:
X_train_flat = X_train.reshape(len(X_train), 784)
X_val_flat   = X_val.reshape(len(X_val), 784)
X_test_flat  = X_test.reshape(len(X_test), 784)

# 3. CNN version: keep the 28x28 grid and add a channel dimension of 1 (grayscale)

- In reshape(len(X_train), 28, 28, 1), the final 1 is the channel dimension: one channel, grayscale.

In [ ]:
X_train_cnn = X_train.reshape(len(X_train), 28, 28, 1)
X_val_cnn   = X_val.reshape(len(X_val), 28, 28, 1)
X_test_cnn  = X_test.reshape(len(X_test), 28, 28, 1)

# 4. Check the results

In [ ]:
print("Pixel range:", X_train.min(), "to", X_train.max())
print("Flat shape: ", X_train_flat.shape)
print("CNN shape:  ", X_train_cnn.shape)

In [ ]:
assert X_train_flat.shape == (50000, 784)
assert X_val_flat.shape == (10000, 784)
assert X_test_flat.shape == (10000, 784)
assert X_train_flat.dtype == np.float32
assert 0.0 <= X_train_flat.min() <= X_train_flat.max() <= 1.0
assert set(np.unique(y_train)) == set(range(10))
print('Train:', X_train_flat.shape, 'Validation:', X_val_flat.shape, 'Test:', X_test_flat.shape)
print('Training label counts:', np.bincount(y_train))

# Trees

In [ ]:
EXPERIMENTS = [
    {'name': 'A', 'n_estimators': 50, 'max_depth': 20},
    {'name': 'B', 'n_estimators': 100, 'max_depth': 20},
    {'name': 'C', 'n_estimators': 100, 'max_depth': None},
]

def build_rf(n_estimators, max_depth):
    return RandomForestClassifier(
        n_estimators=n_estimators,  # number of trees that vote
        max_depth=max_depth,        # None means "no limit"
        random_state=SEED, n_jobs=-1,
    )

# Train w time

# Same steps as experiment A so we use function not to repeat ourselves

In [ ]:
trial_results = []
trial_models = {}

def run_rf(config):
    model = build_rf(config['n_estimators'], config['max_depth'])
    start = time.perf_counter()
    model.fit(X_train_flat, y_train)
    train_seconds = time.perf_counter() - start
    val_accuracy = float(accuracy_score(y_val, model.predict(X_val_flat)))
    train_accuracy = float(accuracy_score(y_train, model.predict(X_train_flat)))
    row = {
        **config,
        'val_accuracy': val_accuracy,
        'train_accuracy': train_accuracy,
        'train_seconds': float(train_seconds),
        'tree_nodes': int(sum(tree.tree_.node_count for tree in model.estimators_)),
        'observed_max_depth': int(max(tree.tree_.max_depth for tree in model.estimators_)),
    }
    trial_models[config['name']] = model
    print(f"Trial {config['name']}: validation accuracy {val_accuracy:.4%}; training {train_seconds:.1f}s")
    return row

In [ ]:
trial_results.append(run_rf(EXPERIMENTS[0]))

changing the number of trees

In [ ]:
for config in EXPERIMENTS[1:]:
    trial_results.append(run_rf(config))
results = pd.DataFrame(trial_results)
print(results.to_string(index=False))

# Compare validation results

In [ ]:
positions = np.arange(len(results))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions - 0.18, results['train_accuracy'], width=0.36, label='Training')
ax.bar(positions + 0.18, results['val_accuracy'], width=0.36, label='Validation')
ax.set_xticks(positions, results['name'])
ax.set_xlabel('Experiment')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_title('Random Forest — training and validation accuracy')
ax.legend()
fig.tight_layout()
plt.show()  # display it

# Save the selected model

In [ ]:
# Rank original configuration dictionaries so unlimited depth stays None rather than NaN.
score_by_name = {row['name']: row['val_accuracy'] for row in trial_results}
selected_config = sorted(
    EXPERIMENTS,
    key=lambda config: (-score_by_name[config['name']], config['n_estimators'],
                        float('inf') if config['max_depth'] is None else config['max_depth'],
                        config['name']),
)[0]
winner = next(row for row in trial_results if row['name'] == selected_config['name'])
best_rf = trial_models[selected_config['name']]
FINAL_PATH = MODEL_DIR / 'rf_final.joblib'
joblib.dump(best_rf, FINAL_PATH, compress=3)
print('Selected configuration:', selected_config)
print('Validation accuracy:', winner['val_accuracy'])
print('Saved compressed model:', FINAL_PATH)

# Evaluate

In [ ]:
load_start = time.perf_counter()
rf_loaded = joblib.load(FINAL_PATH)
load_seconds = time.perf_counter() - load_start

# Warm up using validation data, leaving test evaluation to the next prediction.
_ = rf_loaded.predict_proba(X_val_flat[:1])
start = time.perf_counter()
test_probs = rf_loaded.predict_proba(X_test_flat)
inference_seconds = time.perf_counter() - start
assert test_probs.shape == (10000, 10)
assert np.isfinite(test_probs).all()
assert np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-5)
test_pred = rf_loaded.classes_[np.argmax(test_probs, axis=1)]
test_accuracy = float(accuracy_score(y_test, test_pred))
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, test_pred, average='macro', zero_division=0,
)
# Predictive cross-entropy excludes any L2 weight penalty.
true_class_probs = test_probs[np.arange(len(y_test)), y_test]
test_crossentropy = float(-np.log(np.clip(true_class_probs, 1e-7, 1.0)).mean())
report = classification_report(y_test, test_pred, labels=list(range(10)), output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, test_pred, labels=list(range(10)))

print(f'Test accuracy: {test_accuracy:.4%}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Predictive cross-entropy: {test_crossentropy:.4f}')
print(f'Model size: {FINAL_PATH.stat().st_size / (1024 ** 2):.2f} MiB')
print(f'Loading: {load_seconds:.2f}s; 10,000 predictions after warm-up: {inference_seconds:.2f}s')
print(classification_report(y_test, test_pred, labels=list(range(10)), digits=4, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Selected Random Forest — test confusion matrix')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'rf_confusion_matrix.png', dpi=150)
plt.show()  # display it

wrong = np.flatnonzero(test_pred != y_test)[:12]
if len(wrong):
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, wrong):
        ax.imshow(X_test_raw[index], cmap='gray')
        ax.set_title(f'True: {y_test[index]} / Pred: {test_pred[index]}')
    fig.tight_layout()
    plt.show()  # display it
else:
    print('No test errors in this run.')

# Verify reload and update model comparison

In [ ]:
before_reload = best_rf.predict_proba(X_val_flat[:32])
after_reload = rf_loaded.predict_proba(X_val_flat[:32])
np.testing.assert_allclose(before_reload, after_reload, rtol=1e-7, atol=1e-8)
assert rf_loaded.n_features_in_ == 784
assert np.array_equal(rf_loaded.classes_, np.arange(10))
print('Reload check passed; input (N,784), classes 0–9.')

In [ ]:
comparison_row = {
    'model': 'Random Forest',
    'selected_trial': str(winner['name']),
    'validation_accuracy': float(winner['val_accuracy']),
    'test_accuracy': test_accuracy,
    'macro_f1': float(macro_f1),
    'file_size_mib': FINAL_PATH.stat().st_size / (1024 ** 2),
}
comparison_path = PROJECT_ROOT / 'results' / 'comparison.csv'
previous = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if not previous.empty:
    previous = previous.loc[previous['model'] != comparison_row['model']]
comparison = pd.concat([previous, pd.DataFrame([comparison_row])], ignore_index=True)
comparison = comparison[list(comparison_row)].sort_values('model').reset_index(drop=True)
comparison.to_csv(comparison_path, index=False)
print(pd.DataFrame([comparison_row]).to_string(index=False))

# Release the non-winning forests.
del trial_models, best_rf

# Recorded experiment results

These measurements describe the experiments used to select the included saved model. They are recorded results, not execution outputs from rerunning this notebook.

| Trial | Validation accuracy | Fit seconds |
| --- | --- | --- |
| A | 96.31% | 2.19 |
| B | 96.58% | 4.23 |
| C | 96.65% | 4.33 |

Selected trial: **C**. Test accuracy: **96.95%**; macro F1: **0.9693**.

[Test confusion matrix](report_assets/rf_confusion_matrix.png).

See [the report](report.md) for the full configuration comparison and model recommendation. If you rerun training, update this recorded summary and the report with the new outputs.